# 02. БВС-4: перекрёстное подтверждение lidar / radar / camera, слепота сенсоров и причины помех

**Данные только из ТЗ:** БВС-4 (`sensor_type`, `distance`, `bearing`, `elevation`, `size_estimate`, `detection_confidence`, `sensor_health_status`; сообщения **только при обнаружении**), плюс БВС-5, НСУ-3, НСУ-2 (зоны), БВС-1 (`lat/lon` для проверки зоны) и системные часы (ночь по высоте Солнца).

## Идея
1. **Настоящее препятствие видят несколько сенсоров согласованно** (время ±0,25 с, пеленг ±4°, дальность ±4 м). **Фантом помехи видит один.** Доля *неподтверждённых* детекций сенсора - признак помехи именно этого сенсора: lidar - осадки, пыль, дым; camera - ночные огни, лазер; radar - град.
2. **Слепоту сенсора** (туман, запотевание, лазер) можно выявить только событийно: по дальности первого обнаружения реального препятствия относительно radar (радар не слепнет от тумана). Поток при слепоте **молчит** (ноутбук 00), поэтому нужен именно этот косвенный способ.
3. **Контекст**: ночь (по `timestamp`, `lat/lon`) и активная зона НСУ-2 (объявленное лазерное шоу).

## Что делаем
| Блок | Задача | Метод |
|---|---|---|
| A | проверить идею подтверждения | доля неподтверждённых фантомов и истинных детекций по сенсорам и сценариям |
| B | причина помех / условий | градиентный бустинг по окну признаков + подтверждение смены класса; абляции по источникам |
| C | слепота lidar / camera | монитор дальности первого обнаружения; задержка и её зависимость от частоты препятствий |

> **Честность оценки.** Фантомы и дальности сенсоров заданы генератором (ноутбук 00), поэтому разделимость классов выше, чем будет в реальности. Все метрики - проверка метода и компромиссов, а не точность на борту.

In [ ]:
import sys, pathlib, warnings, time
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, roc_auc_score
from sklearn.inspection import permutation_importance

from src.eval_bvs4 import (KINDS4, CLASSES9, TGT9, build_cache4, build_rows4, eval_blind)
from src.eval_bvs5 import events_from_alerts, dwell_states, segments
from src.detectors_bvs4 import (Bvs4Features, Condition4Detector, BlindnessMonitor, BVS4_FEATS, F4, EXTRA)
from src.detectors_bvs5 import SkWrap, FEATS
from src.streams import DT
plt.rcParams["figure.dpi"] = 110; pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
t0 = time.time()
test = build_cache4(KINDS4, range(400, 408))
print("тестовых сценариев:", len(test), "| время", round(time.time() - t0), "с")

## A. Работает ли перекрёстное подтверждение?
Признак `corr` ставится, если другой сенсор дал детекцию в окне ±0,25 с, ±4° по пеленгу и ±4 м по дальности (решение принимается через 0,5 с, чтобы учесть джиттер доставки). Метка `_phantom` из генератора **используется только здесь для оценки**. Считаем по сенсорам: какую долю фантомов подтверждение отсеивает (полнота отсева) и какую долю истинных детекций ошибочно считает неподтверждёнными.

In [ ]:
rows = []
for c_ in test:
    fe = Bvs4Features(c_["epoch"], keep_log=True)
    for m in c_["msgs"]: fe.feed(m)
    for d in fe.log: rows.append(dict(kind=c_["kind"], sensor=d["s"], phantom=bool(d["ph"]), cor=d["corr"]))
lg = pd.DataFrame(rows)
tab = lg.groupby(["sensor", "phantom"]).agg(n=("cor", "size"), доля_подтверждённых=("cor", "mean")).round(3).unstack("phantom"); display(tab)
ph = lg[lg.phantom]; tr_ = lg[~lg.phantom]
res = pd.DataFrame({"полнота отсева фантомов": 1 - ph.groupby("sensor")["cor"].mean(), "истинные ошибочно неподтверждены": 1 - tr_.groupby("sensor")["cor"].mean()}).round(3); display(res)
bk = tr_.groupby(["kind", "sensor"])["cor"].mean().unstack().round(2); display(bk.loc[["clear", "rain", "dust", "fog_cloud", "laser", "lens_fog"]])

Подтверждение отсеивает почти все фантомы (в генераторе они независимы между сенсорами, поэтому это верхняя оценка). **Цена - слепота:** в тумане остаются без пары уже *истинные* детекции **radar** (lidar и camera их не видят): доля подтверждённых у radar падает до ≈ 0,7. Это учитывается дальше: в классификаторе есть и счётчики, и молчание сенсора (`gap`), а монитор слепоты использует именно такие случаи.

## B. Причина помех и условий
Окно 10 с по БВС-4: на сенсор - число детекций `n`, неподтверждённых `u`, доля малых (< 0,5 м) и близких (< 15 м), средняя уверенность, доли статусов `DEGRADED` и число `FAIL`, пауза с последнего сообщения. К ним добавляются 18 признаков БВС-5/НСУ-3 (ноутбук 01) и контекст (ночь, зона НСУ-2, отношения дальностей lidar/camera к radar). Классы: `dry, rain, snow, fog_cloud, aerosol, hail, laser, night_city` (приоритет: laser > hail > snow > rain > fog_cloud > aerosol > night_city).

In [ ]:
t0 = time.time()
train = build_cache4(KINDS4, range(300, 312)); Xtr, ytr, _ = build_rows4(train); del train          # обучающий кэш освобождаем сразу (память)
Xte, yte, mte = build_rows4(test)
shift_c = build_cache4(KINDS4, range(500, 506), obstacle_rate=1 / 90)
for c_ in shift_c:                                  # смена условий: редкие препятствия + смещение датчиков T/RH
    for m in c_["msgs"]:
        if m["stream"] == "BVS5": m["p"] = dict(m["p"]); m["p"]["temperature"] += 0.5; m["p"]["humidity"] = float(np.clip(m["p"]["humidity"] + 4.0, 1, 100))
Xsh, ysh, msh = build_rows4(shift_c); del shift_c
print("строк:", Xtr.shape, Xte.shape, Xsh.shape, "| время", round(time.time() - t0), "с")
sets = {"БВС-5 + НСУ-3": list(range(18)), "+ окно БВС-4": list(range(18 + 24)), "+ контекст (ночь, зона, дальности)": list(range(len(BVS4_FEATS))), "только БВС-4 + контекст": list(range(18, len(BVS4_FEATS)))}
models = {}; rows = []
for name, cols in sets.items():
    m = HistGradientBoostingClassifier(max_iter=150, random_state=0).fit(Xtr[:, cols], ytr); models[name] = (m, cols); r = {"набор признаков": name}
    for dn, (X, y) in (("тест", (Xte, yte)), ("смена условий", (Xsh, ysh))):
        p = m.predict(X[:, cols]); r[f"macroF1 {dn}"] = f1_score(y, p, average="macro"); r[f"acc {dn}"] = accuracy_score(y, p)
    rows.append(r)
tab = pd.DataFrame(rows).set_index("набор признаков").round(3); display(tab)
pc = pd.DataFrame({n: pd.Series(f1_score(yte, m.predict(Xte[:, cols]), average=None, labels=CLASSES9), index=CLASSES9) for n, (m, cols) in models.items()}).round(2); display(pc)

In [ ]:
full, fcols = models["+ контекст (ночь, зона, дальности)"]; b5, b5c = models["БВС-5 + НСУ-3"]
fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
for a, (name, m, cols) in zip(axs, (("БВС-5 + НСУ-3", b5, b5c), ("со всеми признаками БВС-4", full, fcols))):
    cm = confusion_matrix(yte, m.predict(Xte[:, cols]), labels=CLASSES9, normalize="true"); a.imshow(cm, vmin=0, vmax=1, cmap="Blues")
    a.set_xticks(range(8), CLASSES9, rotation=45, ha="right", fontsize=8); a.set_yticks(range(8), CLASSES9, fontsize=8); a.set_title(name, fontsize=9)
    for (i, j), v in np.ndenumerate(cm): a.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="k" if v < .6 else "w")
plt.tight_layout(); plt.show()
rs = np.random.default_rng(0); idx = rs.choice(len(Xte), 6000, replace=False)
pi = permutation_importance(full, Xte[idx][:, fcols], yte[idx], n_repeats=3, random_state=0, scoring="f1_macro")
imp = pd.Series(pi.importances_mean, index=BVS4_FEATS).sort_values().tail(15); imp.plot.barh(figsize=(5.6, 4.6)); plt.xlabel("падение macro-F1"); plt.tight_layout(); plt.show()

In [ ]:
# дешёвые одно-признаковые базовые правила: AUROC по кадрам окна
rows = []
for feat, pos, name in (("u_lidar", ["rain", "snow", "aerosol", "hail"], "u_lidar → осадки/аэрозоль/град"), ("u_camera", ["laser", "night_city"], "u_camera → лазер/ночные огни"),
                        ("fail_camera", ["laser"], "fail_camera → лазер"), ("u_radar", ["hail"], "u_radar → град")):
    j = BVS4_FEATS.index(feat); y = np.isin(yte, pos); rows.append(dict(правило=name, AUROC=roc_auc_score(y, Xte[:, j])))
display(pd.DataFrame(rows).set_index("правило").round(3))

### Событийная оценка с подтверждением смены класса
Как в ноутбуке 01: смена состояния принимается после `dwell` одинаковых предсказаний подряд. Перебор dwell - офлайн по предсказаниям (ниже проверка, что онлайн-детектор даёт тот же результат). Целевые классы: событие считается обнаруженным, если алерт нужного класса пришёл в окне [onset − 30 с; конец + 30 с].

In [ ]:
def per_scenario(m, cols, X, meta, n):
    cls = m.predict(X[:, cols]); by = {i: ([], []) for i in range(n)}
    for (ci, t), c_ in zip(meta, cls): by[ci][0].append(c_); by[ci][1].append(t)
    return by
def sweep(name, m, cols, X, meta, cache):
    by = per_scenario(m, cols, X, meta, len(cache)); rows = []
    for dw in (1, 3, 5, 8):
        ev, sw = events_from_alerts(cache, [dwell_states(by[i][0], by[i][1], dw) for i in range(len(cache))], tgt=TGT9)
        rows.append(dict(набор=name, dwell=dw, recall_средний=(ev.recall * ev.событий).sum() / ev.событий.sum(), ложных_в_час=ev.ложных_в_час.sum(), переключений_в_час=sw))
    return rows
sw = pd.DataFrame(sweep("БВС-5 + НСУ-3", b5, b5c, Xte, mte, test) + sweep("со всеми признаками БВС-4", full, fcols, Xte, mte, test)).round(2); display(sw.set_index(["набор", "dwell"]))
DW = 3
for name, (m, cols) in (("БВС-5 + НСУ-3", (b5, b5c)), ("со всеми признаками БВС-4", (full, fcols))):
    by = per_scenario(m, cols, Xte, mte, len(test)); ev, swh = events_from_alerts(test, [dwell_states(by[i][0], by[i][1], DW) for i in range(len(test))], tgt=TGT9)
    print(f"{name}, dwell={DW}: переключений в час = {swh:.1f}"); display(ev.round(2))

In [ ]:
by = per_scenario(full, fcols, Xte, mte, len(test)); ok = True
for i in range(0, len(test), 12):
    d = Condition4Detector(SkWrap(full), epoch=test[i]["epoch"], dwell=DW, cols=fcols); d.reset(); tr = []; prev = d.state
    for m in test[i]["msgs"]:
        if m["stream"] in d.streams:
            d.on_message(m)
            if d.state != prev: tr.append((m["t_gen"], d.state)); prev = d.state
    ok &= (tr == dwell_states(by[i][0], by[i][1], DW))
print("онлайн-детектор совпадает с офлайн-подсчётом:", ok)
def timing(make, cache, n_scen=4):
    ts = []
    for c_ in cache[:n_scen]:
        d = make(c_["epoch"]); d.reset()
        for m in c_["msgs"]:
            if m["stream"] in d.streams:
                t1 = time.perf_counter_ns(); d.on_message(m); ts.append((time.perf_counter_ns() - t1) / 1e3)
    a = np.array(ts); return dict(сообщений=len(a), p50_мкс=np.percentile(a, 50), p99_мкс=np.percentile(a, 99), max_мкс=a.max())
rows = [dict(детектор="Bvs4Features (только извлечение признаков)", **timing(lambda ep: type("X", (), dict(streams=("BVS1", "BVS4", "BVS5", "NSU2", "NSU3"), reset=lambda s: s.fe.reset(), on_message=lambda s, m: [s.fe.feed(m)] and [], fe=Bvs4Features(ep)))(), test)),
        dict(детектор="Condition4 + HGB (все потоки)", **timing(lambda ep: Condition4Detector(SkWrap(full), epoch=ep, dwell=DW, cols=fcols), test)),
        dict(детектор="BlindnessMonitor", **timing(lambda ep: BlindnessMonitor(epoch=ep), test))]
tm = pd.DataFrame(rows).set_index("детектор").round(1); tm["p99_мс при замедлении ×50"] = (tm.p99_мкс * 50 / 1000).round(2); display(tm)

## C. Слепота lidar и camera
Монитор: на каждом треке реального препятствия (≥ 8 детекций radar, первая дальность ≥ 100 м) сравнивает дальность **подтверждённого** первого обнаружения lidar и camera с radar; отношение сглаживается (EWMA, α = 0,5). Слепота - EWMA ниже порога. Пороги **раздельные**: у камеры нормальная дальность короче (отношение ≈ 0,8), у lidar ≈ 1,0; при общем пороге флаг камеры «залипает» при частых препятствиях. Истина слепоты (по скрытому состоянию): lidar - дальность < 100 м, camera - дальность < 80 м (туман, запотевание, лазер). Пороги подбираются на отдельных зёрнах.

In [ ]:
tune = build_cache4(KINDS4, range(200, 206)); rows = []
for thr in (0.5, 0.6, 0.7, 0.8):
    r = eval_blind(lambda ep: BlindnessMonitor(thr=thr, thr_cam=0.5, epoch=ep), tune)
    rows.append(dict(сенсор="lidar", thr=thr, recall=r.loc["lidar", "recall"], задержка_с=r.loc["lidar", "задержка_медиана_с"], ложных_в_час=r.loc["lidar", "ложных_в_час"]))
for thr in (0.3, 0.4, 0.5, 0.6):
    r = eval_blind(lambda ep: BlindnessMonitor(thr=0.7, thr_cam=thr, epoch=ep), tune)
    rows.append(dict(сенсор="camera", thr=thr, recall=r.loc["camera", "recall"], задержка_с=r.loc["camera", "задержка_медиана_с"], ложных_в_час=r.loc["camera", "ложных_в_час"]))
bt = pd.DataFrame(rows).round(2); display(bt.set_index(["сенсор", "thr"]))
def pick(sen):
    d = bt[(bt.сенсор == sen) & (bt.ложных_в_час <= 0.5)].sort_values(["recall", "thr"], ascending=[False, True]); return float(d.thr.iloc[0]) if len(d) else float(bt[bt.сенсор == sen].thr.min())
THR_L, THR_C = pick("lidar"), pick("camera"); print("выбраны пороги: lidar", THR_L, "| camera", THR_C, "(критерий: максимум recall при ≤ 0,5 ложных включений в час)")
mk = lambda ep: BlindnessMonitor(thr=THR_L, thr_cam=THR_C, epoch=ep)
del tune
rt = eval_blind(mk, test); display(rt.round(2))
rows = []
for kind in ("fog_cloud", "dust", "smoke", "lens_fog", "laser"):
    sub = [c_ for c_ in test if c_["kind"] == kind]
    for sen in ("lidar", "camera"):
        r = eval_blind(mk, sub).loc[sen]
        if r["событий"]: rows.append(dict(сценарий=kind, сенсор=sen, событий=int(r["событий"]), recall=r["recall"], задержка_медиана_с=r["задержка_медиана_с"]))
display(pd.DataFrame(rows).set_index(["сценарий", "сенсор"]).round(2))

### Зависимость от частоты препятствий
Слепоту видно **только на треках реальных препятствий**, поэтому задержка обнаружения определяется тем, как часто они встречаются (в генераторе по умолчанию в среднем раз в 45 с).

In [ ]:
rows = []; kinds_b = ["clear", "fog_cloud", "dust", "lens_fog", "laser"]
for rate, lab in ((1 / 90, "1 раз в 90 с"), (1 / 45, "1 раз в 45 с"), (1 / 20, "1 раз в 20 с")):
    ch = build_cache4(kinds_b, range(600, 606), obstacle_rate=rate); r = eval_blind(mk, ch)
    for sen in ("lidar", "camera"):
        rows.append(dict(препятствия=lab, сенсор=sen, recall=r.loc[sen, "recall"], задержка_медиана_с=r.loc[sen, "задержка_медиана_с"], ложных_в_час=r.loc[sen, "ложных_в_час"]))
sr = pd.DataFrame(rows).set_index(["препятствия", "сенсор"]).round(2); display(sr)
c_ = next(x for x in test if x["kind"] == "fog_cloud"); fe = Bvs4Features(c_["epoch"]); tt, rl, rc = [], [], []
for m in c_["msgs"]:
    if fe.feed(m) is not None: tt.append(m["t_gen"]); rl.append(fe.ratio["lidar"]); rc.append(fe.ratio["camera"])
fig, ax = plt.subplots(figsize=(9, 3.6)); ax.plot(tt, rl, label="lidar / radar"); ax.plot(tt, rc, label="camera / radar"); ax.axhline(THR_L, c="C0", ls=":"); ax.axhline(THR_C, c="C1", ls=":")
for (s0, s1) in segments(c_["blind"]["lidar"]): ax.axvspan(s0, s1, color="r", alpha=.12, label="истинная слепота lidar")
ax.legend(fontsize=8); ax.set_xlabel("t, с"); ax.set_ylabel("EWMA отношения дальности первого обнаружения"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## Выход модуля
Выход - алерт причины помех (БВС-3) и **вектор доверия сенсоров** (для модуля мониторинга достоверности и НСУ-4/5). Значение `alert_type` для погоды и оптических угроз в ТЗ отсутствует (согласовать).

In [ ]:
def sensor_trust(cause, blind_lidar, blind_cam):
    t = {"lidar": 1.0, "radar": 1.0, "camera": 1.0}
    if cause in ("rain", "snow", "aerosol"): t["lidar"] = 0.5
    if cause in ("laser", "night_city"): t["camera"] = 0.4
    if cause == "hail": t["radar"] = 0.7
    if blind_lidar: t["lidar"] = 0.1
    if blind_cam: t["camera"] = 0.1
    return t
def bvs4_alert(t, cause, p, blind_lidar=False, blind_cam=False, uav_id="UAV-001"):
    act = {"rain": "lower_lidar_confidence; DROR_filter_if_cloud_available", "snow": "lower_lidar_confidence", "aerosol": "lower_lidar_and_camera_confidence; rely_on_radar",
           "fog_cloud": "rely_on_radar; slow_down", "hail": "emergency_land_or_RTH", "laser": "ignore_camera; switch_to_radar_lidar; request_zone_check", "night_city": "raise_camera_threshold"}[cause]
    return {"alert_id": f"b4-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": 4 if cause in ("hail", "laser") else 3, "description": "BVS4_" + cause.upper(),
            "sensor_source": "BVS4+BVS5+NSU3", "confidence": round(float(p), 2), "timestamp": int(t * 1000), "recommended_action": act, "sensor_trust": sensor_trust(cause, blind_lidar, blind_cam)}
display(bvs4_alert(120.0, "laser", 0.87, blind_cam=True))

## 8. Выводы и ограничения
**Перекрёстное подтверждение работает, но это верхняя оценка.** Подтверждение lidar/radar/camera отсеивает ≥ 99,8 % фантомов; ошибочно неподтверждёнными оказываются лишь 0-3 % истинных детекций radar. В генераторе фантомы независимы между сенсорами, в реальности возможны согласованные помехи (например, сильный ливень виден и lidar, и камере), поэтому отсев будет хуже. В тумане неподтверждёнными становятся истинные детекции radar (подтверждено ≈ 71 %), потому что lidar и camera слепы.

**Причина помех и условий.**
- Макро-F1 на тесте: **0,72** (БВС-5 + НСУ-3) → **0,87** (с окном БВС-4 и контекстом); на смене условий (редкие препятствия, смещение датчиков T/RH): 0,60 → 0,79-0,80. Только БВС-4 с контекстом даёт 0,66.
- Выигрыш по классам (F1): лазер 0,02 → 0,54, ночные огни 0,60 → 0,86, град 0,63 → 0,77, аэрозоль 0,89 → 0,98; для дождя, снега и тумана прирост мал (они и так определяются по БВС-5 и НСУ-3).
- Дешёвые одно-признаковые правила: доля неподтверждённых детекций lidar различает «осадки/аэрозоль/град» с AUROC 0,99; неподтверждённые детекции camera - «лазер/ночные огни» (0,90); статус `FAIL` камеры - лазер (0,83); неподтверждённые детекции radar - град (0,85).
- С подтверждением смены класса (dwell = 3, ≈ 3 с): полнота ≈ 0,99 (по классам 0,93-1,0), ложные тревоги **1,4 в час** (без БВС-4: 8,3 в час), переключений состояния 16,8 в час (без БВС-4: 31,8); медианная задержка 2-5 с. Без БВС-4 **лазер не обнаруживается вовсе** (recall 0).
- Контекст ночи и зоны НСУ-2 заметного прироста не дал (зона объявлена лишь в половине лазерных сценариев).

**Слепота сенсоров (признак, которого поток сам не даёт).**
- Слепота **lidar**: recall 1,0, ложных включений нет, но **задержка ≈ 40 с**; слепота **camera**: recall ≈ 0,5, задержка ≈ 90 с (при условии ≤ 0,5 ложных включений в час). Для коротких лазерных событий (20-40 с) монитор слепоты бесполезен (recall 0): их ловит классификатор по `FAIL` и неподтверждённым детекциям камеры.
- Причина задержки - слепоту видно **только на треках реальных препятствий**: при встрече препятствий раз в 90 с медианная задержка для lidar ≈ 93 с, раз в 45 с ≈ 22 с, раз в 20 с ≈ 20 с. Поток БВС-4 в штатном режиме молчит, и это принципиальный предел. Для устранения нужен периодический сигнал состояния сенсоров (heartbeat по `sensor_health_status` независимо от обнаружений), которого в ТЗ нет: это важно зафиксировать как требование/риск.
- Отдельные пороги для lidar (0,7) и camera (0,3): у камеры нормальная дальность короче, и общий порог приводил к «залипанию» флага.

**Время.** Извлечение признаков: p50 ≈ 5 мкс, p99 ≈ 97 мкс на сообщение (редкие выбросы до 3 мс); монитор слепоты: p99 ≈ 88 мкс; классификатор с бустингом построчно: p99 ≈ 1,1 мс (при замедлении ×50 ≈ 53 мс). Бустинг вызывается только на сообщении БВС-5 (1 Гц), остальные потоки обрабатываются за десятки микросекунд; для борта бустинг нужно сжать или вызывать асинхронно.

**Ограничения.**
- Фантомы, дальности сенсоров в тумане, статусы здоровья - допущения генератора; в реальном потоке интенсивность и независимость фантомов неизвестны.
- `point_cloud_ref` не используется: формат интенсивности в ТЗ не указан.
- Радар считается нечувствительным к погоде, кроме градовых фантомов; сильный дождь в реальности ослабляет и радар.
- Задержка слепоты зависит от частоты препятствий; в пустом воздушном пространстве слепота не выявляется вовсе.

**Дальше:** ноутбук 03 (мощность и устойчивость по БВС-5, БВС-1, БВС-7: налипание снега, намокание, град по `error_codes` и attitude), затем слияние видимости (`visibility_m`, БВС-4, БВС-5) и интегральный индекс пригодности условий.